In [ ]:
import json
import time

import requests

BASE_URL = "https://www.wheeloratings.com/src/xscores"
DELAY_SECONDS = 1

In [ ]:
# Download the index file and get the unique last 2 characters of each player's WebsiteId
index = requests.get(f"{BASE_URL}/xscores_data.json", timeout=30)
index.raise_for_status()
index_data = index.json()

last_2 = sorted({w[-2:] for w in index_data["Lookups"]["Player"][0]["WebsiteId"]})
print(len(last_2), last_2)

In [ ]:
# Scrape {last_2}.json for each value, 1 second apart
all_data = {}
failed = {}

for i, val in enumerate(last_2):
    url = f"{BASE_URL}/{val}.json"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        all_data[val] = r.json()
    except (requests.RequestException, ValueError) as e:
        failed[val] = str(e)
        print(f"FAILED {url}: {e}")
    if i < len(last_2) - 1:
        time.sleep(DELAY_SECONDS)

print(f"scraped {len(all_data)}/{len(last_2)}, failed {len(failed)}")

In [ ]:
# Save everything as a single JSON file keyed by the last-2 value
with open("../data/xscores_all_data.json", "w") as f:
    json.dump(all_data, f)
print("saved ../data/xscores_all_data.json")